# 🚦 Event-Driven Congestion / Gridlock Intelligence Platform
## Bengaluru Traffic Command Center — Hackathon Prototype
### Stack: OSMnx 2.x · NetworkX · Folium · Gemini AI · Pandas

**Pipeline:**  
Data → EDA → Feature Engineering → Congestion Engine → Resource Recommender → Alternate Routes → AI Reports → Dashboard

## 📦 CELL 1 — Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    'osmnx', 'networkx', 'folium', 'google-generativeai',
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy'
]

for pkg in packages:
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', pkg, '-q', '--upgrade'],
            capture_output=True, text=True, timeout=120
        )
        print(f'✅ {pkg} ready' if result.returncode == 0 else f'⚠️  {pkg}: {result.stderr[:80]}')
    except Exception as e:
        print(f'❌ {pkg}: {e}')

print('\n✅ All packages ready!')

## 🔑 CELL 2 — Imports & Configuration

In [ ]:
import os, sys, json, warnings, math, time, traceback
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

import osmnx as ox
import networkx as nx
import folium
from folium.plugins import MarkerCluster, HeatMap
import google.generativeai as genai

# ── API KEYS ──────────────────────────────────────────────────────────────────
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY') or os.environ.get('GEMINI_API_KEY')
MAPMYINDIA_KEY = os.environ.get('MAPMYINDIA_KEY')

# ── PATHS ─────────────────────────────────────────────────────────────────────
BASE_DIR   = Path(r'C:\hackthongrid')
DATA_FILE  = BASE_DIR / 'Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv'
OUTPUT_DIR = BASE_DIR
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Configure Gemini ──────────────────────────────────────────────────────────
GEMINI_AVAILABLE = False
try:
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-1.5-flash')
    _test = gemini_model.generate_content('Reply with only the word: OK')
    if _test.text:
        GEMINI_AVAILABLE = True
        print(f'✅ Gemini 1.5 Flash connected: {_test.text.strip()}')
except Exception as e:
    print(f'⚠️  Gemini unavailable (fallback mode): {e}')

# ── OSMnx 2.x settings ────────────────────────────────────────────────────────
ox.settings.log_console = False
ox.settings.use_cache   = True

# ── Plot theme ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d',   'axes.labelcolor': '#e6edf3',
    'xtick.color': '#e6edf3',      'ytick.color': '#e6edf3',
    'text.color': '#e6edf3',       'grid.color': '#21262d',
    'legend.facecolor': '#161b22', 'legend.edgecolor': '#30363d',
})

print(f'\n✅ Imports loaded!')
print(f'   OSMnx   : {ox.__version__}')
print(f'   Folium  : {folium.__version__}')
print(f'   Pandas  : {pd.__version__}')
print(f'   Gemini  : {"Available" if GEMINI_AVAILABLE else "Fallback mode"}')

## 📊 CELL 3 — Data Pipeline

In [ ]:
print('📂 Loading dataset...')
df_raw = pd.read_csv(DATA_FILE, low_memory=False)
print(f'   Raw shape: {df_raw.shape}')

USEFUL_COLS = [
    'id', 'event_type', 'latitude', 'longitude',
    'endlatitude', 'endlongitude', 'address', 'event_cause',
    'requires_road_closure', 'start_datetime', 'end_datetime',
    'status', 'corridor', 'priority', 'description',
    'veh_type', 'police_station', 'zone', 'junction',
    'created_date', 'closed_datetime'
]
available = [c for c in USEFUL_COLS if c in df_raw.columns]
df = df_raw[available].copy()

# Clean coordinates
for col in ['latitude', 'longitude', 'endlatitude', 'endlongitude']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df.loc[df[col] == 0, col] = np.nan

# Parse datetimes
for col in ['start_datetime', 'end_datetime', 'closed_datetime', 'created_date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

# Clean categoricals
df['event_cause'] = df['event_cause'].fillna('unknown').str.strip()
df['event_type']  = df['event_type'].fillna('unplanned').str.strip()
df['corridor']    = df['corridor'].fillna('Non-corridor').str.strip()
df['priority']    = df['priority'].fillna('Low').str.strip()
df['status']      = df['status'].fillna('unknown').str.strip()

df = df.dropna(axis=1, how='all')
df_geo = df.dropna(subset=['latitude', 'longitude']).copy()

print(f'   Working shape  : {df.shape}')
print(f'   Geo-valid shape: {df_geo.shape}')
print(f'\n✅ Data pipeline complete!')

## 📈 CELL 4 — EDA

In [ ]:
print('=' * 60)
print('  EXPLORATORY DATA ANALYSIS')
print('=' * 60)
print(f'\n📐 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Geo-valid: {df_geo.shape[0]:,} rows')

print('\n🔍 Missing values:')
miss = df.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
for col, cnt in miss.items():
    pct = cnt / len(df) * 100
    print(f'   {col:<25} {cnt:>5} ({pct:5.1f}%)')

print('\n📊 Event Type:')
print(df['event_type'].value_counts().to_string())

print('\n🔎 Event Cause (Top 15):')
print(df['event_cause'].value_counts().head(15).to_string())

print('\n🛣️  Corridor (Top 15):')
print(df['corridor'].value_counts().head(15).to_string())

print('\n⚡ Priority:')
print(df['priority'].value_counts().to_string())

print('\n🚧 Road Closure:')
print(df['requires_road_closure'].value_counts().to_string())

df_time = df.dropna(subset=['start_datetime']).copy()
df_time['hour'] = df_time['start_datetime'].dt.hour
hourly = df_time['hour'].value_counts().sort_index()
peak_h = hourly.idxmax()
print(f'\n⏰ Peak hour: {peak_h}:00 ({hourly[peak_h]} events)')

print('\n📍 Top 10 Police Stations:')
if 'police_station' in df.columns:
    print(df['police_station'].value_counts().head(10).to_string())

print('\n✅ EDA complete!')

## 📊 CELL 4b — EDA Charts

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 16))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('🚦 Bengaluru Traffic Event — EDA Dashboard', fontsize=18,
             color='#58a6ff', fontweight='bold', y=1.01)

def styled_bar(ax, labels, values, title, color='#58a6ff', top_n=12):
    if len(labels) > top_n:
        labels, values = labels[:top_n], values[:top_n]
    colors = [color if i < 3 else '#30363d' for i in range(len(labels))]
    bars = ax.barh(range(len(labels)), values, color=colors)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_title(title, color='#e6edf3', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{int(val):,}', va='center', fontsize=8, color='#e6edf3')

# 1. Event cause
cause_vc = df['event_cause'].value_counts()
styled_bar(axes[0,0], cause_vc.index.tolist(), cause_vc.values.tolist(), '🔎 Event Cause', '#58a6ff')

# 2. Corridor
corr_vc = df['corridor'].value_counts()
styled_bar(axes[0,1], corr_vc.index.tolist(), corr_vc.values.tolist(), '🛣️ Corridor', '#3fb950')

# 3. Hourly trend
df_th = df.dropna(subset=['start_datetime']).copy()
df_th['hour'] = df_th['start_datetime'].dt.hour
hourly = df_th['hour'].value_counts().sort_index()
axes[1,0].plot(hourly.index, hourly.values, color='#f78166', linewidth=2, marker='o', markersize=4)
axes[1,0].fill_between(hourly.index, hourly.values, alpha=0.3, color='#f78166')
axes[1,0].axvspan(7, 10, alpha=0.15, color='#ffa657', label='Morning Rush')
axes[1,0].axvspan(17, 20, alpha=0.15, color='#d2a8ff', label='Evening Rush')
axes[1,0].legend(fontsize=8)
axes[1,0].set_xlabel('Hour of Day')
axes[1,0].set_title('⏰ Hourly Event Frequency', color='#e6edf3', fontsize=12, fontweight='bold')

# 4. Priority pie
prio_vc = df['priority'].value_counts()
wedge_colors = ['#f78166', '#ffa657', '#3fb950', '#58a6ff']
axes[1,1].pie(prio_vc.values, labels=prio_vc.index,
              colors=wedge_colors[:len(prio_vc)],
              autopct='%1.1f%%', textprops={'color': '#e6edf3'}, startangle=90)
axes[1,1].set_title('⚡ Priority Distribution', color='#e6edf3', fontsize=12, fontweight='bold')

# 5. Day of week
df_th['dow'] = df_th['start_datetime'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_vc = df_th['dow'].value_counts().reindex(dow_order, fill_value=0)
axes[2,0].bar(range(7), dow_vc.values,
              color=['#f78166' if d in ['Saturday','Sunday'] else '#58a6ff' for d in dow_order])
axes[2,0].set_xticks(range(7))
axes[2,0].set_xticklabels([d[:3] for d in dow_order])
axes[2,0].set_title('📅 Day-of-Week', color='#e6edf3', fontsize=12, fontweight='bold')
axes[2,0].legend(handles=[
    mpatches.Patch(color='#f78166', label='Weekend'),
    mpatches.Patch(color='#58a6ff', label='Weekday')
], fontsize=8)

# 6. Road closure
rc_vc = df['requires_road_closure'].value_counts()
axes[2,1].bar(rc_vc.index.astype(str), rc_vc.values, color=['#3fb950','#f78166'])
axes[2,1].set_title('🚧 Road Closure Required', color='#e6edf3', fontsize=12, fontweight='bold')
for i, v in enumerate(rc_vc.values):
    axes[2,1].text(i, v + 10, f'{v:,}', ha='center', fontsize=10, color='#e6edf3')

plt.tight_layout()
eda_path = OUTPUT_DIR / 'eda_dashboard.png'
plt.savefig(eda_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'✅ EDA chart saved → {eda_path}')

## 🔧 CELL 5 — Feature Engineering

In [ ]:
print('🔧 Building features...')
df_feat = df_geo.copy()

# Time features
df_feat['start_hour'] = df_feat['start_datetime'].dt.hour.fillna(12).astype(int)
df_feat['weekday']    = df_feat['start_datetime'].dt.weekday.fillna(0).astype(int)
df_feat['month']      = df_feat['start_datetime'].dt.month.fillna(1).astype(int)
df_feat['is_weekend'] = (df_feat['weekday'] >= 5).astype(int)
df_feat['rush_hour']  = df_feat['start_hour'].apply(lambda h: 1 if (7<=h<=10) or (17<=h<=20) else 0)
df_feat['peak_hour']  = df_feat['start_hour'].apply(lambda h: 1 if h in [8,9,18,19] else 0)
df_feat['night_time'] = df_feat['start_hour'].apply(lambda h: 1 if h>=22 or h<=6 else 0)

# Event duration
def compute_duration(row):
    try:
        if pd.notna(row.get('closed_datetime')) and pd.notna(row.get('start_datetime')):
            delta = (row['closed_datetime'] - row['start_datetime']).total_seconds() / 60
            return max(0, delta) if delta < 1440 else np.nan
    except:
        pass
    return np.nan

df_feat['event_duration_mins'] = df_feat.apply(compute_duration, axis=1)

# Corridor risk score
HIGH_RISK_CORRIDORS = {
    'ORR East 1': 9, 'ORR East 2': 8, 'ORR North 1': 8, 'ORR North 2': 7,
    'ORR West 1': 7, 'Bellary Road 1': 8, 'Bellary Road 2': 7,
    'Hosur Road': 8, 'Tumkur Road': 7, 'Mysore Road': 7,
    'Bannerghata Road': 7, 'CBD 1': 9, 'CBD 2': 9,
    'Old Madras Road': 7, 'Magadi Road': 6,
    'West of Chord Road': 6, 'Non-corridor': 3,
}
df_feat['corridor_risk_score'] = df_feat['corridor'].map(HIGH_RISK_CORRIDORS).fillna(5)

# Cause frequency score
cause_counts = df_feat['event_cause'].value_counts(normalize=True) * 100
cause_norm   = (cause_counts - cause_counts.min()) / (cause_counts.max() - cause_counts.min() + 1e-9) * 10
df_feat['cause_frequency_score'] = df_feat['event_cause'].map(cause_norm).fillna(5)

# Severity/impact score (rule-based, no leakage)
CAUSE_SEV = {
    'accident': 9, 'vehicle_breakdown': 5, 'water_logging': 7,
    'construction': 6, 'protest': 8, 'vip_movement': 8,
    'public_event': 7, 'tree_fall': 6, 'procession': 7,
    'congestion': 8, 'pot_holes': 4, 'road_conditions': 4,
    'others': 3, 'unknown': 3,
}
df_feat['cause_severity']   = df_feat['event_cause'].map(CAUSE_SEV).fillna(4)
df_feat['priority_score']   = df_feat['priority'].map({'High':3,'Medium':2,'Low':1}).fillna(1)
df_feat['closure_flag']     = df_feat['requires_road_closure'].astype(int)

df_feat['impact_score'] = (
    df_feat['cause_severity']       * 0.40 +
    df_feat['corridor_risk_score']  * 0.30 +
    df_feat['priority_score']       * 0.10 * 3 +
    df_feat['rush_hour']            * 0.10 * 10 +
    df_feat['closure_flag']         * 0.10 * 10
).clip(0, 10).round(2)

clean_path = OUTPUT_DIR / 'cleaned_events.csv'
df_feat.to_csv(clean_path, index=False)

print(f'✅ Features complete. Shape: {df_feat.shape}')
print(f'   Impact score — mean: {df_feat["impact_score"].mean():.2f}, max: {df_feat["impact_score"].max():.2f}')
print(f'✅ Cleaned dataset saved → {clean_path}')

## 🔴 CELL 6 — Congestion Impact Engine

In [ ]:
def compute_congestion_score(event: dict) -> dict:
    """Rule-based congestion scoring. Input: event dict. Output: score dict."""
    score = 0

    cause_pts = {
        'accident': 30, 'water_logging': 25, 'protest': 28,
        'vip_movement': 26, 'procession': 24, 'public_event': 22,
        'construction': 18, 'tree_fall': 16, 'congestion': 20,
        'vehicle_breakdown': 12, 'pot_holes': 8, 'road_conditions': 6,
        'others': 5, 'unknown': 5,
    }
    cause = str(event.get('event_cause', 'unknown')).lower()
    score += cause_pts.get(cause, 5)

    if event.get('requires_road_closure') in [True, 'True', 'true', 1, '1']:
        score += 20

    corridor_pts = {
        'ORR East 1': 20, 'ORR East 2': 18, 'CBD 1': 20, 'CBD 2': 18,
        'ORR North 1': 16, 'ORR North 2': 15, 'Bellary Road 1': 16,
        'Bellary Road 2': 14, 'Hosur Road': 16, 'Tumkur Road': 14,
        'Mysore Road': 14, 'Bannerghata Road': 14, 'Old Madras Road': 12,
        'West of Chord Road': 12, 'Magadi Road': 10, 'Non-corridor': 4,
    }
    corridor = str(event.get('corridor', 'Non-corridor'))
    score += corridor_pts.get(corridor, 8)

    score += {'High': 10, 'Medium': 6, 'Low': 2}.get(str(event.get('priority','Low')), 2)

    hour = int(event.get('start_hour', 12))
    if hour in [8, 9, 18, 19]:              score += 10
    elif 7 <= hour <= 10 or 17 <= hour <= 20: score += 7
    elif 22 <= hour or hour <= 6:            score += 2
    else:                                    score += 4

    if int(event.get('weekday', 0)) >= 5:
        score = int(score * 0.85)

    if str(event.get('event_type','unplanned')) == 'unplanned':
        score += 5

    score = min(100, max(0, score))

    if   score >= 75: risk = 'CRITICAL'
    elif score >= 55: risk = 'HIGH'
    elif score >= 35: risk = 'MODERATE'
    else:             risk = 'LOW'

    severity_map = {'CRITICAL':'Gridlock','HIGH':'Severe','MODERATE':'Moderate','LOW':'Minor'}
    urgency_map  = {
        'CRITICAL': 'IMMEDIATE (< 5 min)',
        'HIGH':     'URGENT (< 15 min)',
        'MODERATE': 'STANDARD (< 30 min)',
        'LOW':      'ROUTINE (< 60 min)'
    }
    return {
        'congestion_score':  score,
        'risk_level':        risk,
        'severity_category': severity_map[risk],
        'response_urgency':  urgency_map[risk],
    }


# Apply to dataset
print('🔴 Scoring congestion for all events...')
cong_results = df_feat.apply(lambda r: compute_congestion_score(r.to_dict()), axis=1)
cong_df = pd.DataFrame(cong_results.tolist())
df_feat = pd.concat([df_feat.reset_index(drop=True), cong_df], axis=1)

print(df_feat['risk_level'].value_counts().to_string())
print(f'\n   Mean congestion score: {df_feat["congestion_score"].mean():.1f}')
print('\n✅ Congestion engine complete!')

## 👮 CELL 7 — Resource Recommendation Engine

In [ ]:
def recommend_resources(event: dict) -> dict:
    """Recommend officers, barricades, patrol vehicles based on event type."""
    cause   = str(event.get('event_cause', 'unknown')).lower()
    priority= str(event.get('priority', 'Low'))
    corridor= str(event.get('corridor', 'Non-corridor'))
    closure = event.get('requires_road_closure', False) in [True,'True','true',1,'1']
    hour    = int(event.get('start_hour', 12))
    rush    = (7 <= hour <= 10) or (17 <= hour <= 20)
    is_corr = corridor not in ['Non-corridor', 'NULL', '', 'nan']

    configs = {
        'accident':         {'officers':6,  'barricades':8,  'patrol_vehicles':2,
            'plan':'Cordon accident zone. Deploy at intersections. Coordinate with BBMP for towing. Alert emergency services.'},
        'vehicle_breakdown':{'officers':2,  'barricades':4,  'patrol_vehicles':1,
            'plan':'Barricade behind breakdown vehicle. Arrange tow-truck within 30 minutes.'},
        'water_logging':    {'officers':4,  'barricades':6,  'patrol_vehicles':1,
            'plan':'Block waterlogged zone entry. Upstream diversions. Coordinate BWSSB/BBMP drainage.'},
        'construction':     {'officers':3,  'barricades':10, 'patrol_vehicles':1,
            'plan':'Channelized lane. Regulatory signs. Speed monitoring in zone.'},
        'protest':          {'officers':12, 'barricades':15, 'patrol_vehicles':3,
            'plan':'Perimeter barricades. Riot-readiness unit. PCR vans. Pre-divert corridor traffic.'},
        'vip_movement':     {'officers':10, 'barricades':12, 'patrol_vehicles':4,
            'plan':'Cordon route 30 min ahead. Pilot escort. Officers at every junction.'},
        'public_event':     {'officers':8,  'barricades':10, 'patrol_vehicles':2,
            'plan':'Dedicated parking zones. Crowd management. Tow trucks on standby. Diversion 2h before.'},
        'tree_fall':        {'officers':3,  'barricades':6,  'patrol_vehicles':1,
            'plan':'Block lane. Request BBMP/BESCOM tree clearance. Est. 1-3 hours.'},
        'procession':       {'officers':8,  'barricades':12, 'patrol_vehicles':2,
            'plan':'Route escort officers. Diversion at start point. Alert hospitals en route.'},
        'congestion':       {'officers':4,  'barricades':4,  'patrol_vehicles':1,
            'plan':'Officers at gridlock junctions. Manual signal control. Assess contraflow.'},
        'pot_holes':        {'officers':1,  'barricades':4,  'patrol_vehicles':0,
            'plan':'Warning signage. Flag for BBMP urgent repair.'},
        'others':           {'officers':2,  'barricades':3,  'patrol_vehicles':1,
            'plan':'Assess situation. Deploy standard traffic management.'},
    }

    cfg = configs.get(cause, configs['others']).copy()

    mult = 1.0
    if rush:           mult += 0.5
    if closure:        mult += 0.3
    if is_corr:        mult += 0.2
    if priority=='High': mult += 0.2

    officers        = max(1, round(cfg['officers']        * mult))
    barricades      = max(2, round(cfg['barricades']      * mult))
    patrol_vehicles = max(0, round(cfg['patrol_vehicles'] * mult))

    congestion = compute_congestion_score(event)
    score      = congestion['congestion_score']

    if   score >= 75: div_urgency = 'IMMEDIATE — Activate alternate routes NOW'
    elif score >= 55: div_urgency = 'HIGH — Divert within 10 minutes'
    elif score >= 35: div_urgency = 'MODERATE — Prepare alternate routes'
    else:             div_urgency = 'LOW — Monitor and stand by'

    return {
        'risk_level':       congestion['risk_level'],
        'congestion_score': congestion['congestion_score'],
        'severity_category':congestion['severity_category'],
        'officers':         officers,
        'barricades':       barricades,
        'patrol_vehicles':  patrol_vehicles,
        'diversion_urgency':div_urgency,
        'action_plan':      cfg['plan'],
        'response_urgency': congestion['response_urgency'],
    }


# Test
test_event = {
    'event_cause':'accident', 'requires_road_closure':True,
    'corridor':'ORR East 1', 'priority':'High',
    'start_hour':8, 'weekday':2, 'event_type':'unplanned'
}
r = recommend_resources(test_event)
print('👮 Resource Engine Test:')
for k,v in r.items(): print(f'   {k:<22}: {v}')
print('\n✅ Resource engine ready!')

## 🗺️ CELL 8 — Bengaluru Road Graph (OSMnx 2.x)

In [ ]:
print('🗺️  Loading Bengaluru road network (OSMnx 2.x)...')
print('   First run: ~60-90s download + caching. Cached runs: ~5s.')

BENGALURU_GRAPH = None

try:
    BENGALURU_GRAPH = ox.graph_from_place(
        'Bengaluru, Karnataka, India',
        network_type='drive',
        simplify=True
    )
    nodes, edges = ox.graph_to_gdfs(BENGALURU_GRAPH)
    print(f'✅ Graph loaded! Nodes: {len(nodes):,}  Edges: {len(edges):,}')
except Exception as e:
    print(f'⚠️  Full-city graph failed: {e}')
    print('   Trying bbox fallback (central Bengaluru)...')
    try:
        # OSMnx 2.x graph_from_bbox signature: (bbox, **kwargs)
        # bbox = (north, south, east, west)
        BENGALURU_GRAPH = ox.graph_from_bbox(
            bbox=(13.05, 12.88, 77.70, 77.48),
            network_type='drive',
            simplify=True
        )
        nodes, edges = ox.graph_to_gdfs(BENGALURU_GRAPH)
        print(f'✅ Bbox graph: {len(nodes):,} nodes, {len(edges):,} edges')
    except Exception as e2:
        print(f'⚠️  Bbox also failed: {e2}')
        print('   All routing will use simulated fallback.')
        BENGALURU_GRAPH = None

print(f'\n   Graph status: {"✅ OSMnx loaded" if BENGALURU_GRAPH else "⚠️  Simulated fallback"}')

## 🛤️ CELL 9 — Alternate Route Engine

In [ ]:
def _haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = phi2 - phi1
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def _route_distance_km(G, route):
    """Compute route distance manually from graph edge lengths (metres → km)."""
    total = 0.0
    for u, v in zip(route[:-1], route[1:]):
        edata = G.get_edge_data(u, v)
        if edata:
            key = list(edata.keys())[0]
            total += edata[key].get('length', 0)
    return total / 1000.0


def _simulate_route(src_lat, src_lon, dst_lat, dst_lon, label='original'):
    straight_km = _haversine_km(src_lat, src_lon, dst_lat, dst_lon)
    road_km = round(straight_km * 1.35, 2)
    offset = 0.005 if label == 'alternate' else 0.0
    mid_lat = (src_lat + dst_lat) / 2 + offset
    mid_lon = (src_lon + dst_lon) / 2 + offset
    waypoints = [
        (src_lat, src_lon),
        ((src_lat*2 + mid_lat)/3, (src_lon*2 + mid_lon)/3),
        (mid_lat, mid_lon),
        ((mid_lat + dst_lat*2)/3, (mid_lon + dst_lon*2)/3),
        (dst_lat, dst_lon)
    ]
    return waypoints, road_km


def generate_alternate_route(
    source_lat, source_lon,
    destination_lat, destination_lon,
    blocked_lat=None, blocked_lon=None,
    block_radius_m=150
) -> dict:
    """
    Generate original + alternate route.
    Priority: OSMnx 2.x → simulated fallback.
    Never raises — always returns result dict.
    """
    result = {
        'original_route_coords':   [],
        'alternate_route_coords':  [],
        'original_distance_km':    None,
        'alternate_distance_km':   None,
        'extra_distance_km':       None,
        'estimated_delay_minutes': None,
        'route_found':             False,
        'routing_engine':          'none',
        'diversion_recommendation': '',
        'warnings':                [],
    }

    coords = [source_lat, source_lon, destination_lat, destination_lon]
    if any(c is None or (isinstance(c, float) and math.isnan(c)) for c in coords):
        result['warnings'].append('Invalid coordinates')
        result['diversion_recommendation'] = 'Manual route assessment required'
        return result

    # ── Try OSMnx ────────────────────────────────────────────────────────────
    if BENGALURU_GRAPH is not None:
        try:
            # OSMnx 2.x: ox.nearest_nodes(G, X=lon, Y=lat)
            src_node = ox.nearest_nodes(BENGALURU_GRAPH, X=source_lon,      Y=source_lat)
            dst_node = ox.nearest_nodes(BENGALURU_GRAPH, X=destination_lon, Y=destination_lat)

            orig_route = nx.shortest_path(BENGALURU_GRAPH, src_node, dst_node, weight='length')
            orig_coords = [
                (BENGALURU_GRAPH.nodes[n]['y'], BENGALURU_GRAPH.nodes[n]['x'])
                for n in orig_route
            ]
            orig_dist_km = _route_distance_km(BENGALURU_GRAPH, orig_route)

            result['original_route_coords'] = orig_coords
            result['original_distance_km']  = round(orig_dist_km, 2)
            result['route_found']           = True
            result['routing_engine']        = 'OSMnx+NetworkX'

            # Build blocked graph
            G_blocked = BENGALURU_GRAPH.copy()

            if blocked_lat is not None and blocked_lon is not None:
                blocked_nodes = [
                    n for n, d in G_blocked.nodes(data=True)
                    if _haversine_km(blocked_lat, blocked_lon, d.get('y',0), d.get('x',0)) * 1000 <= block_radius_m
                ]
            else:
                mid = len(orig_route) // 2
                blocked_nodes = orig_route[max(0,mid-2):min(len(orig_route),mid+3)]

            G_blocked.remove_nodes_from(blocked_nodes)
            result['warnings'].append(f'Blocked {len(blocked_nodes)} nodes')

            src_b = src_node if src_node in G_blocked.nodes else ox.nearest_nodes(G_blocked, X=source_lon, Y=source_lat)
            dst_b = dst_node if dst_node in G_blocked.nodes else ox.nearest_nodes(G_blocked, X=destination_lon, Y=destination_lat)

            alt_route    = nx.shortest_path(G_blocked, src_b, dst_b, weight='length')
            alt_coords   = [(G_blocked.nodes[n]['y'], G_blocked.nodes[n]['x']) for n in alt_route]
            alt_dist_km  = _route_distance_km(G_blocked, alt_route)

            extra = round(alt_dist_km - orig_dist_km, 2)
            delay = max(0, round(extra * 2.0 + 5, 1))

            result['alternate_route_coords'] = alt_coords
            result['alternate_distance_km']  = round(alt_dist_km, 2)
            result['extra_distance_km']      = extra
            result['estimated_delay_minutes']= delay

            if extra < 0:
                result['diversion_recommendation'] = f'Alternate is shorter by {abs(extra):.1f} km — use alternate.'
            elif extra < 2:
                result['diversion_recommendation'] = f'Alternate adds +{extra:.1f} km (~{delay} min). Recommend diversion.'
            elif extra < 5:
                result['diversion_recommendation'] = f'Alternate +{extra:.1f} km (~{delay} min). Use if blockage > 20 min.'
            else:
                result['diversion_recommendation'] = f'Alternate is significantly longer (+{extra:.1f} km). Clear blockage first.'

            return result

        except nx.NetworkXNoPath:
            result['warnings'].append('No path in OSMnx — using simulated fallback')
        except nx.NodeNotFound:
            result['warnings'].append('Node not found — using simulated fallback')
        except Exception as e:
            result['warnings'].append(f'OSMnx error: {str(e)[:80]} — using simulated fallback')

    # ── Simulated fallback ────────────────────────────────────────────────────
    result['routing_engine'] = 'Simulated'
    result['warnings'].append('⚠️ SIMULATED route (OSMnx unavailable or failed)')

    orig_coords, orig_km = _simulate_route(source_lat, source_lon, destination_lat, destination_lon, 'original')
    alt_coords,  alt_km  = _simulate_route(source_lat, source_lon, destination_lat, destination_lon, 'alternate')
    alt_km = round(alt_km * 1.15, 2)
    extra  = round(alt_km - orig_km, 2)
    delay  = max(0, round(extra * 2.0 + 5, 1))

    result.update({
        'original_route_coords':   orig_coords,
        'alternate_route_coords':  alt_coords,
        'original_distance_km':    orig_km,
        'alternate_distance_km':   alt_km,
        'extra_distance_km':       extra,
        'estimated_delay_minutes': delay,
        'route_found':             True,
        'diversion_recommendation': f'[SIMULATED] Alternate +{extra:.1f} km (~{delay} min). Deploy officers at upstream junction.',
    })
    return result


# Test
print('🛤️  Testing routing engine...')
test_route = generate_alternate_route(
    source_lat=12.9716, source_lon=77.5946,
    destination_lat=12.9352, destination_lon=77.6245,
    blocked_lat=12.9550, blocked_lon=77.6100, block_radius_m=150
)
print(f"   Engine   : {test_route['routing_engine']}")
print(f"   Original : {test_route['original_distance_km']} km")
print(f"   Alternate: {test_route['alternate_distance_km']} km")
print(f"   Extra    : {test_route['extra_distance_km']} km | Delay: {test_route['estimated_delay_minutes']} min")
print(f"   Found    : {test_route['route_found']}")
for w in test_route.get('warnings',[]): print(f'   ⚠️  {w}')
print('\n✅ Routing engine ready!')

## 🤖 CELL 10 — Gemini AI Modules

In [ ]:
def _fallback_incident_report(event, resources, route):
    cause    = event.get('event_cause','unknown').replace('_',' ').title()
    location = event.get('address', event.get('police_station','Bengaluru'))
    risk     = resources.get('risk_level','MODERATE')
    score    = resources.get('congestion_score',50)
    text = f"""BENGALURU TRAFFIC POLICE — INCIDENT REPORT
Date/Time : {datetime.now().strftime('%d %B %Y, %H:%M IST')}
Incident  : {cause} at {location}
Risk Level: {risk} (Congestion Score: {score}/100)

RESOURCES DEPLOYED:
  Officers        : {resources.get('officers','N/A')}
  Barricades      : {resources.get('barricades','N/A')}
  Patrol Vehicles : {resources.get('patrol_vehicles','N/A')}

TRAFFIC IMPACT:
  Diversion Urgency    : {resources.get('diversion_urgency','N/A')}
  Extra Route Distance : {route.get('extra_distance_km','N/A')} km
  Estimated Delay      : {route.get('estimated_delay_minutes','N/A')} minutes
  Routing Engine       : {route.get('routing_engine','N/A')}

ACTION PLAN:
  {resources.get('action_plan','Standard protocols activated.')}

STATUS: Active | Response: {resources.get('response_urgency','STANDARD')}
"""
    return {
        'report_text':     text,
        'generated_by':    'fallback-template',
        'incident_id':     f"INC-{datetime.now().strftime('%Y%m%d%H%M%S')}",
        'risk_level':      risk,
        'congestion_score':score,
    }


def _fallback_public_advisory(event, resources, route):
    cause  = event.get('event_cause','unknown').replace('_',' ').title()
    loc    = event.get('address', event.get('police_station','Bengaluru'))
    risk   = resources.get('risk_level','MODERATE')
    delay  = route.get('estimated_delay_minutes',10)
    text = f"""🚦 TRAFFIC ADVISORY — BENGALURU TRAFFIC POLICE

⚠️  {cause} reported near {loc}

📌 SITUATION: {cause} causing {risk} disruption. Delay: ~{delay} minutes.

🔁 DIVERSION: {route.get('diversion_recommendation','Please use alternate routes.')}

✅ CITIZEN ADVICE:
   1. Avoid the area for next {int(delay)+15} minutes
   2. Follow officer directions at diversions
   3. Keep left — do not obstruct emergency vehicles
   4. Use Waze / Google Maps for live updates

📞 Police: 100 | Emergency: 112
🐦 @BlrCityTrafficPolice for live updates
⏰ Issued: {datetime.now().strftime('%d %b %Y %H:%M IST')}
"""
    return {'advisory_text':text,'generated_by':'fallback-template','severity':risk,'delay_minutes':delay}


def generate_incident_report(event, resources, route) -> dict:
    if not GEMINI_AVAILABLE:
        return _fallback_incident_report(event, resources, route)
    prompt = f"""You are an official Bengaluru Traffic Police incident report writer.
Write a structured formal incident report. Use ONLY the data provided below. Do NOT invent facts.

cause: {event.get('event_cause')}, location: {event.get('address',event.get('police_station'))}
corridor: {event.get('corridor')}, priority: {event.get('priority')}
risk_level: {resources.get('risk_level')}, congestion_score: {resources.get('congestion_score')}/100
officers: {resources.get('officers')}, barricades: {resources.get('barricades')}, vehicles: {resources.get('patrol_vehicles')}
original_km: {route.get('original_distance_km')}, alt_km: {route.get('alternate_distance_km')}
extra_km: {route.get('extra_distance_km')}, delay_min: {route.get('estimated_delay_minutes')}

Include: 1) Incident summary 2) Resources deployed 3) Traffic impact 4) Action plan 5) Status
Under 300 words. Plain text only."""
    try:
        resp = gemini_model.generate_content(prompt)
        return {
            'report_text':     resp.text,
            'generated_by':    'gemini-1.5-flash',
            'incident_id':     f"INC-{datetime.now().strftime('%Y%m%d%H%M%S')}",
            'risk_level':      resources.get('risk_level'),
            'congestion_score':resources.get('congestion_score'),
        }
    except Exception as e:
        print(f'   ⚠️ Gemini quota/error ({e}), using fallback')
        return _fallback_incident_report(event, resources, route)


def generate_public_advisory(event, resources, route) -> dict:
    if not GEMINI_AVAILABLE:
        return _fallback_public_advisory(event, resources, route)
    prompt = f"""You are the social media manager for Bengaluru Traffic Police.
Write a concise public traffic advisory. Use ONLY the facts below. No invented streets or distances.

cause: {event.get('event_cause','').replace('_',' ')}
location: {event.get('address', event.get('police_station','Bengaluru'))}
risk: {resources.get('risk_level')}, score: {resources.get('congestion_score')}/100
delay: {route.get('estimated_delay_minutes')} minutes
diversion: {route.get('diversion_recommendation')}

Format: 🚦 headline, situation (1-2 sentences), 3-4 bullet advice points, helplines (100, 112), timestamp.
Under 200 words. Tone: clear, factual, calm."""
    try:
        resp = gemini_model.generate_content(prompt)
        return {'advisory_text':resp.text,'generated_by':'gemini-1.5-flash',
                'severity':resources.get('risk_level'),'delay_minutes':route.get('estimated_delay_minutes')}
    except Exception as e:
        print(f'   ⚠️ Gemini quota/error ({e}), using fallback')
        return _fallback_public_advisory(event, resources, route)


print(f'🤖 AI modules ready. Mode: {"Gemini 1.5 Flash" if GEMINI_AVAILABLE else "Fallback templates"}')

## 🎬 CELL 11 — End-to-End Pipeline

In [ ]:
def run_demo_event(event_dict: dict, verbose=True) -> dict:
    """Full pipeline: scoring → resources → routing → AI reports."""
    name = event_dict.get('scenario_name', 'Demo')
    if verbose:
        print(f'\n{"═"*60}\n  🎬 {name}\n{"═"*60}')

    # 1. Congestion
    congestion = compute_congestion_score(event_dict)
    event_dict.update(congestion)
    if verbose: print(f'  🔴 Congestion: {congestion["congestion_score"]}/100 ({congestion["risk_level"]})')

    # 2. Resources
    resources = recommend_resources(event_dict)
    if verbose: print(f'  👮 Resources: {resources["officers"]}👮 {resources["barricades"]}🚧 {resources["patrol_vehicles"]}🚓')

    # 3. Routing
    route = generate_alternate_route(
        source_lat=event_dict.get('source_lat', event_dict.get('latitude', 12.9716)),
        source_lon=event_dict.get('source_lon', event_dict.get('longitude', 77.5946)),
        destination_lat=event_dict.get('destination_lat', 12.9352),
        destination_lon=event_dict.get('destination_lon', 77.6245),
        blocked_lat=event_dict.get('latitude'),
        blocked_lon=event_dict.get('longitude'),
        block_radius_m=event_dict.get('block_radius_m', 150)
    )
    if verbose:
        print(f'  🛤️  Route [{route["routing_engine"]}]: '
              f'{route["original_distance_km"]}km → {route["alternate_distance_km"]}km '
              f'(+{route["extra_distance_km"]}km, ~{route["estimated_delay_minutes"]}min)')

    # 4. Incident report
    if verbose: print('  🤖 Generating incident report...')
    incident_report = generate_incident_report(event_dict, resources, route)

    # 5. Public advisory
    if verbose: print('  📢 Generating public advisory...')
    public_advisory = generate_public_advisory(event_dict, resources, route)

    if verbose: print(f'  ✅ Done! AI: {incident_report["generated_by"]}')

    return {
        'scenario_name':   name,
        'event':           event_dict,
        'congestion':      congestion,
        'resources':       resources,
        'route':           route,
        'incident_report': incident_report,
        'public_advisory': public_advisory,
    }

print('✅ Pipeline function run_demo_event() defined!')

## 🏙️ CELL 12 — 10 Demo Scenarios

In [ ]:
SCENARIOS = [
    {
        'scenario_name':'S01 — Major Accident on ORR',
        'event_cause':'accident', 'event_type':'unplanned', 'requires_road_closure':True,
        'latitude':12.9352, 'longitude':77.6900,
        'source_lat':12.9716, 'source_lon':77.5946, 'destination_lat':12.9136, 'destination_lon':77.7100,
        'corridor':'ORR East 1', 'priority':'High', 'start_hour':8, 'weekday':1,
        'address':'Outer Ring Road, Marathahalli Junction, Bengaluru', 'police_station':'HAL Old Airport',
    },
    {
        'scenario_name':'S02 — BMTC Bus Breakdown on Hosur Road',
        'event_cause':'vehicle_breakdown', 'event_type':'unplanned', 'requires_road_closure':False,
        'latitude':12.9071, 'longitude':77.6286,
        'source_lat':12.9352, 'source_lon':77.6245, 'destination_lat':12.8560, 'destination_lon':77.6645,
        'corridor':'Hosur Road', 'priority':'High', 'start_hour':18, 'weekday':3,
        'address':'Hosur Road, Vivekananda Circle, Bommanahalli', 'police_station':'Madiwala',
    },
    {
        'scenario_name':'S03 — Severe Waterlogging at Underpass',
        'event_cause':'water_logging', 'event_type':'unplanned', 'requires_road_closure':True,
        'latitude':13.0000, 'longitude':77.6814,
        'source_lat':13.0190, 'source_lon':77.6556, 'destination_lat':12.9760, 'destination_lon':77.7100,
        'corridor':'ORR East 2', 'priority':'High', 'start_hour':7, 'weekday':2,
        'address':'Whitefield Road, ITI Data Center Underpass, Dooravani Nagar', 'police_station':'K.R. Pura',
    },
    {
        'scenario_name':'S04 — Political Protest at Town Hall',
        'event_cause':'protest', 'event_type':'planned', 'requires_road_closure':True,
        'latitude':12.9738, 'longitude':77.5965,
        'source_lat':12.9850, 'source_lon':77.5988, 'destination_lat':12.9600, 'destination_lon':77.6020,
        'corridor':'CBD 1', 'priority':'High', 'start_hour':10, 'weekday':4,
        'address':'Town Hall, Ambedkar Veedhi, Cubbon Park, Bengaluru', 'police_station':'Cubbon Park',
    },
    {
        'scenario_name':'S05 — VIP Movement on Bellary Road',
        'event_cause':'vip_movement', 'event_type':'planned', 'requires_road_closure':True,
        'latitude':13.0000, 'longitude':77.5841,
        'source_lat':12.9850, 'source_lon':77.5988, 'destination_lat':13.0420, 'destination_lon':77.5947,
        'corridor':'Bellary Road 1', 'priority':'High', 'start_hour':9, 'weekday':0,
        'address':'Bellary Road, Sadashiva Nagar to Hebbal Flyover', 'police_station':'Sadashivanagar',
    },
    {
        'scenario_name':'S06 — Metro Pillar Construction on ORR',
        'event_cause':'construction', 'event_type':'planned', 'requires_road_closure':False,
        'latitude':12.9695, 'longitude':77.7007,
        'source_lat':12.9760, 'source_lon':77.6950, 'destination_lat':12.9465, 'destination_lon':77.6987,
        'corridor':'ORR East 2', 'priority':'High', 'start_hour':7, 'weekday':1,
        'address':'Outer Ring Road, Karthik Nagar, Marathahalli Metro Station', 'police_station':'HAL Old Airport',
    },
    {
        'scenario_name':'S07 — IPL Match at Chinnaswamy Stadium',
        'event_cause':'public_event', 'event_type':'planned', 'requires_road_closure':False,
        'latitude':12.9793, 'longitude':77.5996,
        'source_lat':12.9850, 'source_lon':77.5988, 'destination_lat':12.9650, 'destination_lon':77.6000,
        'corridor':'CBD 2', 'priority':'High', 'start_hour':17, 'weekday':5,
        'address':'MG Road, Cubbon Park area, Bengaluru', 'police_station':'Cubbon Park',
    },
    {
        'scenario_name':'S08 — Tree Fall Blocking Sankey Road',
        'event_cause':'tree_fall', 'event_type':'unplanned', 'requires_road_closure':True,
        'latitude':13.0062, 'longitude':77.5794,
        'source_lat':13.0190, 'source_lon':77.5700, 'destination_lat':12.9900, 'destination_lon':77.5800,
        'corridor':'Bellary Road 1', 'priority':'Low', 'start_hour':20, 'weekday':3,
        'address':'Sankey Road, Bashyam Circle, Sadashiva Nagar', 'police_station':'Sadashivanagar',
    },
    {
        'scenario_name':'S09 — Religious Procession on Mysore Road',
        'event_cause':'procession', 'event_type':'planned', 'requires_road_closure':True,
        'latitude':12.9441, 'longitude':77.5274,
        'source_lat':12.9600, 'source_lon':77.5400, 'destination_lat':12.9200, 'destination_lon':77.5000,
        'corridor':'Mysore Road', 'priority':'High', 'start_hour':6, 'weekday':6,
        'address':'Mysore Road, Rajarajeshwari Junction, Nayandahalli', 'police_station':'Byatarayanapura',
    },
    {
        'scenario_name':'S10 — Public Gathering at Lalbagh',
        'event_cause':'public_event', 'event_type':'planned', 'requires_road_closure':False,
        'latitude':12.9507, 'longitude':77.5848,
        'source_lat':12.9600, 'source_lon':77.5700, 'destination_lat':12.9300, 'destination_lon':77.5900,
        'corridor':'Non-corridor', 'priority':'Medium', 'start_hour':8, 'weekday':6,
        'address':'Lalbagh Botanical Garden, V V Puram, Bengaluru', 'police_station':'V.V.Puram (C.Pet)',
    },
]

print('🎬 Running 10 demo scenarios...\n')
demo_results = []

for i, scenario in enumerate(SCENARIOS, 1):
    try:
        result = run_demo_event(scenario.copy(), verbose=True)
        demo_results.append(result)
    except Exception as e:
        print(f'   ❌ Scenario {i} failed: {e}')
        traceback.print_exc()

print(f'\n✅ {len(demo_results)}/10 scenarios completed!')

## 🗺️ CELL 13 — Folium Map Dashboard

In [ ]:
print('🗺️  Building Folium dashboard...')

m = folium.Map(location=[12.9716, 77.5946], zoom_start=12, tiles='CartoDB dark_matter')

layer_incidents = folium.FeatureGroup(name='🔴 Incidents', show=True)
layer_orig      = folium.FeatureGroup(name='🔵 Original Routes', show=True)
layer_alt       = folium.FeatureGroup(name='🟢 Alternate Routes', show=True)
layer_heatmap   = folium.FeatureGroup(name='🌡️ Congestion Heatmap', show=False)

RISK_COLORS = {'CRITICAL':'red','HIGH':'orange','MODERATE':'blue','LOW':'green'}

for result in demo_results:
    ev   = result['event']
    res  = result['resources']
    rte  = result['route']
    name = result['scenario_name']
    lat  = ev.get('latitude')
    lon  = ev.get('longitude')
    if lat is None or lon is None: continue

    risk_color = RISK_COLORS.get(res.get('risk_level','LOW'),'blue')
    score      = res.get('congestion_score', 0)

    popup_html = f"""
    <div style="font-family:monospace;background:#1a1a2e;color:#e0e0e0;padding:12px;
                border-radius:8px;border:1px solid #58a6ff;min-width:280px">
        <h3 style="color:#58a6ff;margin:0 0 8px 0;font-size:13px">{name}</h3>
        <hr style="border-color:#30363d;margin:4px 0">
        <b style="color:#f78166">Risk:</b> {res.get('risk_level','N/A')} &nbsp;
        <b>Score:</b> <span style="color:#ffa657">{score}/100</span><br>
        <b>Cause:</b> {ev.get('event_cause','N/A').replace('_',' ').title()}<br>
        <b>Corridor:</b> {ev.get('corridor','N/A')}<br>
        <hr style="border-color:#30363d;margin:4px 0">
        <b style="color:#3fb950">👮 Officers:</b> {res.get('officers','N/A')}&emsp;
        <b>Barricades:</b> {res.get('barricades','N/A')}&emsp;
        <b>Vehicles:</b> {res.get('patrol_vehicles','N/A')}<br>
        <hr style="border-color:#30363d;margin:4px 0">
        <b>🛤️ Original:</b> {rte.get('original_distance_km','N/A')} km<br>
        <b style="color:#3fb950">🔄 Alternate:</b> {rte.get('alternate_distance_km','N/A')} km
        (+{rte.get('extra_distance_km','N/A')} km, ~{rte.get('estimated_delay_minutes','N/A')} min)<br>
        <b>Engine:</b> {rte.get('routing_engine','N/A')}
    </div>"""

    folium.CircleMarker(
        location=[lat, lon], radius=14+(score//15),
        color=risk_color, fill=True, fill_color=risk_color, fill_opacity=0.7, weight=2,
        popup=folium.Popup(popup_html, max_width=320),
        tooltip=f"{name} | {res.get('risk_level')} | {score}/100"
    ).add_to(layer_incidents)

    orig_coords = rte.get('original_route_coords', [])
    if len(orig_coords) >= 2:
        folium.PolyLine(orig_coords, color='#4d9de0', weight=3, opacity=0.7,
                        tooltip=f'Original: {rte.get("original_distance_km")} km').add_to(layer_orig)

    alt_coords = rte.get('alternate_route_coords', [])
    if len(alt_coords) >= 2:
        folium.PolyLine(alt_coords, color='#3fb950', weight=3, opacity=0.7, dash_array='6 3',
                        tooltip=f'Alt: {rte.get("alternate_distance_km")} km (+{rte.get("extra_distance_km")} km)').add_to(layer_alt)

    for mk_lat, mk_lon, icon, color, tip in [
        (ev.get('source_lat', lat), ev.get('source_lon', lon), 'play', 'blue', f'Start: {name}'),
        (ev.get('destination_lat', lat), ev.get('destination_lon', lon), 'flag', 'green', f'End: {name}'),
    ]:
        folium.Marker([mk_lat, mk_lon],
                      icon=folium.Icon(color=color, icon=icon, prefix='fa'),
                      tooltip=tip).add_to(layer_orig)

# Dataset-level heatmap
dataset_heat = df_feat[['latitude','longitude','congestion_score']].dropna().sample(
    min(2000, len(df_feat)), random_state=42)
heat_pts = [[r['latitude'], r['longitude'], r['congestion_score']/100]
            for _, r in dataset_heat.iterrows()]
HeatMap(heat_pts, radius=12, blur=8, min_opacity=0.3,
        gradient={'0.4':'blue','0.65':'lime','1':'red'}).add_to(layer_heatmap)

for layer in [layer_incidents, layer_orig, layer_alt, layer_heatmap]:
    layer.add_to(m)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;
            background:#0d1117;border:1px solid #30363d;border-radius:8px;
            padding:12px;font-family:monospace;color:#e6edf3;font-size:12px">
    <b style="color:#58a6ff;font-size:14px">🚦 BLR Traffic Intel</b><br>
    <hr style="border-color:#30363d;margin:6px 0">
    <span style="color:red">●</span> CRITICAL (75-100)<br>
    <span style="color:orange">●</span> HIGH (55-74)<br>
    <span style="color:#4d9de0">●</span> MODERATE (35-54)<br>
    <span style="color:green">●</span> LOW (0-34)<br>
    <hr style="border-color:#30363d;margin:6px 0">
    <span style="color:#4d9de0">—</span> Original Route<br>
    <span style="color:#3fb950">- -</span> Alternate Route
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(position='topright', collapsed=False).add_to(m)

dashboard_path = OUTPUT_DIR / 'traffic_dashboard.html'
m.save(str(dashboard_path))
print(f'\n✅ Dashboard saved → {dashboard_path}')
print(f'   Open: file:///{str(dashboard_path).replace(chr(92), "/")}')

## 📋 CELL 14 — Export Demo Results CSV

In [ ]:
rows = []
for result in demo_results:
    ev, res, rte = result['event'], result['resources'], result['route']
    inc, adv     = result['incident_report'], result['public_advisory']
    rows.append({
        'scenario':               result['scenario_name'],
        'event_cause':            ev.get('event_cause'),
        'corridor':               ev.get('corridor'),
        'priority':               ev.get('priority'),
        'hour':                   ev.get('start_hour'),
        'requires_road_closure':  ev.get('requires_road_closure'),
        'congestion_score':       res.get('congestion_score'),
        'risk_level':             res.get('risk_level'),
        'severity_category':      res.get('severity_category'),
        'response_urgency':       res.get('response_urgency'),
        'officers':               res.get('officers'),
        'barricades':             res.get('barricades'),
        'patrol_vehicles':        res.get('patrol_vehicles'),
        'diversion_urgency':      res.get('diversion_urgency'),
        'routing_engine':         rte.get('routing_engine'),
        'route_found':            rte.get('route_found'),
        'original_distance_km':   rte.get('original_distance_km'),
        'alternate_distance_km':  rte.get('alternate_distance_km'),
        'extra_distance_km':      rte.get('extra_distance_km'),
        'estimated_delay_minutes':rte.get('estimated_delay_minutes'),
        'diversion_recommendation':rte.get('diversion_recommendation','')[:200],
        'incident_report':        inc.get('report_text','')[:300],
        'public_advisory':        adv.get('advisory_text','')[:300],
        'ai_engine':              inc.get('generated_by'),
    })

demo_df = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / 'demo_results.csv'
demo_df.to_csv(csv_path, index=False)
print(f'✅ Demo results saved → {csv_path}')
print(f'   {len(demo_df)} rows, {len(demo_df.columns)} columns')

## 📊 CELL 15 — Final Summary & Readiness Score

In [ ]:
print('\n' + '═'*100)
print('  🏆 FINAL DEMO SUMMARY — Bengaluru Traffic Intelligence Platform')
print('═'*100)

summary = demo_df[[
    'scenario','risk_level','congestion_score',
    'officers','barricades','patrol_vehicles',
    'route_found','extra_distance_km','diversion_urgency'
]].copy()
summary.columns = ['Scenario','Risk','Score','Officers','Barricades','Vehicles',
                   'Route?','Extra km','Diversion']
print(summary.to_string(index=False))
print('═'*100)

# Readiness scorecard
print('\n📋 SYSTEM READINESS SCORECARD:')
checks = {
    'Data Pipeline':              True,
    'EDA Charts Generated':       (OUTPUT_DIR/'eda_dashboard.png').exists(),
    'Feature Engineering':        'impact_score' in df_feat.columns,
    'Congestion Engine':          'congestion_score' in df_feat.columns,
    'Resource Recommender':       True,
    'OSMnx Road Graph':           BENGALURU_GRAPH is not None,
    'Simulated Fallback Routing': True,
    'Gemini AI Reports':          GEMINI_AVAILABLE,
    'Fallback AI Reports':        True,
    '10 Demo Scenarios':          len(demo_results) == 10,
    'Folium Dashboard HTML':      (OUTPUT_DIR/'traffic_dashboard.html').exists(),
    'Demo Results CSV':           (OUTPUT_DIR/'demo_results.csv').exists(),
    'Cleaned Dataset CSV':        (OUTPUT_DIR/'cleaned_events.csv').exists(),
}

passed = sum(v for v in checks.values())
total  = len(checks)
score  = round(passed / total * 10, 1)

print(f'\n  {"Component":<40} Status')
print(f'  {"-"*40} ------')
for k, v in checks.items():
    print(f'  {k:<40} {"✅" if v else "⚠️ "}')

print(f'\n  ┌──────────────────────────────────┐')
print(f'  │  FINAL READINESS SCORE: {score:>4.1f}/10  │')
print(f'  └──────────────────────────────────┘')
print(f'  {passed}/{total} checks passed')

print('\n🎉 DELIVERABLES:')
for label, path in [
    ('Cleaned Events Dataset',   OUTPUT_DIR/'cleaned_events.csv'),
    ('EDA Dashboard PNG',        OUTPUT_DIR/'eda_dashboard.png'),
    ('Traffic Dashboard HTML',   OUTPUT_DIR/'traffic_dashboard.html'),
    ('Demo Results CSV',         OUTPUT_DIR/'demo_results.csv'),
]:
    exists = path.exists()
    size   = f'{path.stat().st_size//1024:,} KB' if exists else 'not found'
    print(f'  {"✅" if exists else "❌"} {label:<35} {path.name} ({size})')

print('\n✅ Hackathon prototype complete!')

## 🖨️ CELL 16 — Print Sample AI Reports

In [ ]:
for result in demo_results[:2]:
    print('\n' + '─'*70)
    print(f'  SCENARIO: {result["scenario_name"]}')
    print('─'*70)
    print('\n📋 INCIDENT REPORT:')
    print(result['incident_report']['report_text'])
    print('\n📢 PUBLIC ADVISORY:')
    print(result['public_advisory']['advisory_text'])

print('\n✅ All done!')